# Olist — Kiểm kê và làm sạch dữ liệu

Chạy **Restart Kernel → Run All** trong VS Code với kernel Python có `pandas`.
Đặt notebook cạnh thư mục `data/raw`, chứa đúng tên 9 CSV Olist.

**Phạm vi:** 1. đọc dữ liệu → 2. kiểm kê và kiểm tra chất lượng → 3. làm sạch → 4. kiểm tra, xuất dữ liệu.

- Giữ ID và mã bưu chính dưới dạng chuỗi.
- Không xóa hàng loạt dòng thiếu hoặc tự điền các ngày giao hàng.
- Giữ văn bản review; tạo cờ có nội dung và cột điểm hợp lệ.
- Chỉ bỏ dòng trùng hoàn toàn ở bản geolocation dùng tra cứu; giữ bảng nguồn.
- Giá trị số/ngày bất thường được đánh dấu; không tự phỏng đoán giá trị thay thế.
- Bản dịch danh mục được ghép để chuẩn hóa tên, không ghép các bảng giao dịch thành bảng phân tích.

Notebook không tạo nhãn giao trễ, biến nghiên cứu, bảng tổng hợp cấp đơn hoặc thực hiện kiểm định/ML.
Các phép đếm và tỷ lệ thiếu chỉ phục vụ kiểm tra chất lượng. Xuất 10 CSV sạch vào `data/processed`; không ghi đè `data/raw`.


In [1]:
# 1. Chuẩn bị và đọc dữ liệu
# Mục đích: Đọc đủ 9 bảng; giữ nguyên ID và mã bưu chính.

from pathlib import Path
from IPython.display import display
import pandas as pd

DATA_DIR = Path("data/raw")
OUTPUT_DIR = Path("data/processed")

files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}
data = {}
for name, filename in files.items():
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(f"Không tìm thấy: {path.resolve()}")
    data[name] = pd.read_csv(path, dtype="string")
    print(f"{name}: {len(data[name]):,} dòng")


customers: 99,441 dòng
orders: 99,441 dòng
items: 112,650 dòng
payments: 103,886 dòng
reviews: 99,224 dòng
products: 32,951 dòng
sellers: 3,095 dòng
geolocation: 1,000,163 dòng
translation: 71 dòng


## 2. Kiểm tra chất lượng trước khi làm sạch

Các cell này chỉ đọc hoặc chuyển kiểu trên biến tạm. Số liệu hiển thị là kết quả
tính từ dữ liệu đang chạy, không phải số được điền sẵn.
`duplicated()` đếm số dòng dư; `duplicated(..., keep=False)` đếm mọi dòng thuộc nhóm lặp.


In [2]:
# 2.1. Tổng quan, missing và trùng dòng
# Mục đích: Xem quy mô từng bảng và các cột thiếu dữ liệu.

display(pd.DataFrame([
    {"table": name, "rows": len(df), "columns": len(df.columns),
     "missing_cells": df.isna().sum().sum(),
     "exact_duplicate_rows": df.duplicated().sum()}
    for name, df in data.items()
]))

for name in ["reviews", "orders", "products"]:
    df = data[name]
    missing = pd.DataFrame({"missing_count": df.isna().sum(),
                            "missing_percent": df.isna().mean().mul(100).round(2)})
    print("\nBảng:", name)
    display(missing.loc[missing["missing_count"].gt(0)])

orders = data["orders"]
date_columns = ["order_approved_at", "order_delivered_carrier_date",
                "order_delivered_customer_date"]
missing_by_status = orders[date_columns].isna().groupby(orders["order_status"]).sum()
missing_by_status.insert(0, "total_orders", orders.groupby("order_status").size())
display(missing_by_status)



Bảng: reviews

Bảng: orders

Bảng: products


,table,rows,columns,missing_cells,exact_duplicate_rows
0,customers,99441,5,0,0
1,orders,99441,8,4908,0
2,items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,products,32951,9,2448,0
6,sellers,3095,4,0,0
7,geolocation,1000163,5,0,261831
8,translation,71,2,0,0


,missing_count,missing_percent
review_comment_title,87656,88.34
review_comment_message,58247,58.70


,missing_count,missing_percent
order_approved_at,160,0.16
order_delivered_carrier_date,1783,1.79
order_delivered_customer_date,2965,2.98


,missing_count,missing_percent
product_category_name,610,1.85
product_name_lenght,610,1.85
product_description_lenght,610,1.85
product_photos_qty,610,1.85
product_weight_g,2,0.01
product_length_cm,2,0.01
product_height_cm,2,0.01
product_width_cm,2,0.01


,total_orders,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,,
approved,2,0,2,2
canceled,625,141,550,619
created,5,5,5,5
delivered,96478,14,2,8
invoiced,314,0,314,314
processing,301,0,301,301
shipped,1107,0,0,1107
unavailable,609,0,609,609


In [3]:
# 2.2. Khóa và liên kết
# Mục đích: Kiểm tra khóa thiếu/lặp và mã không tồn tại ở bảng đích.

key_columns = {
    "customers": ["customer_id"], "orders": ["order_id"],
    "items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id", "order_id"], "products": ["product_id"],
    "sellers": ["seller_id"], "translation": ["product_category_name"],
}
key_report = []
for name, keys in key_columns.items():
    df = data[name]
    key_report.append({"table": name, "key": " + ".join(keys),
                       "missing_key_rows": df[keys].isna().any(axis=1).sum(),
                       "duplicate_key_rows": df.duplicated(keys, keep=False).sum()})
display(pd.DataFrame(key_report))
print("Reviews: số dòng thuộc nhóm review_id lặp:",
      data["reviews"].duplicated("review_id", keep=False).sum())

relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("items", "order_id", "orders", "order_id"),
    ("items", "product_id", "products", "product_id"),
    ("items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "translation", "product_category_name"),
]
relationship_report = []
for child, col, parent, parent_col in relationships:
    values = data[child][col]
    unmatched = values.notna() & ~values.isin(data[parent][parent_col].dropna())
    relationship_report.append({
        "relationship": f"{child}.{col} → {parent}.{parent_col}",
        "missing_rows": values.isna().sum(), "unmatched_rows": unmatched.sum(),
        "unmatched_distinct_values": values[unmatched].nunique(),
    })
display(pd.DataFrame(relationship_report))

category = data["products"]["product_category_name"]
untranslated = category.notna() & ~category.isin(data["translation"]["product_category_name"])
display(category[untranslated].value_counts().to_frame("product_count"))


Reviews: số dòng thuộc nhóm review_id lặp: 1603


,table,key,missing_key_rows,duplicate_key_rows
0,customers,customer_id,0,0
1,orders,order_id,0,0
2,items,order_id + order_item_id,0,0
3,payments,order_id + payment_sequential,0,0
4,reviews,review_id + order_id,0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,translation,product_category_name,0,0


,relationship,missing_rows,unmatched_rows,unmatched_distinct_values
0,orders.customer_id → customers.customer_id,0,0,0
1,items.order_id → orders.order_id,0,0,0
2,items.product_id → products.product_id,0,0,0
3,items.seller_id → sellers.seller_id,0,0,0
4,payments.order_id → orders.order_id,0,0,0
5,reviews.order_id → orders.order_id,0,0,0
6,products.product_category_name → translation.p...,610,13,2


,product_count
product_category_name,
portateis_cozinha_e_preparadores_de_alimentos,10
pc_gamer,3


In [4]:
# 2.3. Geolocation
# Mục đích: Phân biệt trùng cả dòng với nhiều tọa độ chung một mã bưu chính.

geo = data["geolocation"]
zip_column = "geolocation_zip_code_prefix"
display(pd.DataFrame([{
    "total_rows": len(geo), "exact_duplicate_rows": geo.duplicated().sum(),
    "missing_zip_rows": geo[zip_column].isna().sum(),
    "unique_zip_prefixes": geo[zip_column].nunique(),
    "rows_with_repeated_zip": geo.duplicated(zip_column, keep=False).sum(),
}]))

coordinate_report = []
for col, limit in [("geolocation_lat", 90), ("geolocation_lng", 180)]:
    source = geo[col]
    values = pd.to_numeric(source, errors="coerce")
    coordinate_report.append({
        "column": col, "missing_rows": source.isna().sum(),
        "invalid_numeric_rows": (source.notna() & values.isna()).sum(),
        "out_of_range_rows": (values.notna() & ~values.between(-limit, limit)).sum(),
    })
display(pd.DataFrame(coordinate_report))
# Giới hạn toàn cầu chỉ kiểm tra miền giá trị, chưa xác nhận tọa độ đúng vị trí ở Brazil.


,total_rows,exact_duplicate_rows,missing_zip_rows,unique_zip_prefixes,rows_with_repeated_zip
0,1000163,261831,0,19015,999120


,column,missing_rows,invalid_numeric_rows,out_of_range_rows
0,geolocation_lat,0,0,0
1,geolocation_lng,0,0,0


In [5]:
# 2.4. Miền giá trị số
# Mục đích: Kiểm tra giá/phí, số kỳ, điểm review và thông số sản phẩm.

rules = [
    ("items", "price", 0, None, False),
    ("items", "freight_value", 0, None, False),
    ("items", "order_item_id", 1, None, True),
    ("payments", "payment_value", 0, None, False),
    ("payments", "payment_sequential", 1, None, True),
    ("payments", "payment_installments", 1, None, True),
    ("reviews", "review_score", 1, 5, True),
]
report = []
for name, col, low, high, integer in rules:
    source = data[name][col]
    values = pd.to_numeric(source, errors="coerce")
    outside = values.lt(low)
    if high is not None:
        outside = outside | values.gt(high)
    report.append({
        "table": name, "column": col, "missing_rows": source.isna().sum(),
        "invalid_numeric_rows": (source.notna() & values.isna()).sum(),
        "outside_expected_range": outside.sum(),
        "non_integer_rows": values.mod(1).ne(0).where(values.notna(), False).sum() if integer else 0,
    })
display(pd.DataFrame(report))

product_numeric = ["product_name_lenght", "product_description_lenght", "product_photos_qty",
                   "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
report = []
for col in product_numeric:
    source = data["products"][col]
    values = pd.to_numeric(source, errors="coerce")
    report.append({"column": col, "missing_rows": source.isna().sum(),
                   "invalid_numeric_rows": (source.notna() & values.isna()).sum(),
                   "negative_rows": values.lt(0).sum(), "zero_rows": values.eq(0).sum()})
display(pd.DataFrame(report))

installments = pd.to_numeric(data["payments"]["payment_installments"], errors="coerce")
display(data["payments"].loc[installments.lt(1)])
weight = pd.to_numeric(data["products"]["product_weight_g"], errors="coerce")
display(data["products"].loc[weight.eq(0),
        ["product_id", "product_category_name", "product_weight_g",
         "product_length_cm", "product_height_cm", "product_width_cm"]])


,table,column,missing_rows,invalid_numeric_rows,outside_expected_range,non_integer_rows
0,items,price,0,0,0,0
1,items,freight_value,0,0,0,0
2,items,order_item_id,0,0,0,0
3,payments,payment_value,0,0,0,0
4,payments,payment_sequential,0,0,0,0
5,payments,payment_installments,0,0,2,0
6,reviews,review_score,0,0,0,0


,column,missing_rows,invalid_numeric_rows,negative_rows,zero_rows
0,product_name_lenght,610,0,0,0
1,product_description_lenght,610,0,0,0
2,product_photos_qty,610,0,0,0
3,product_weight_g,2,0,0,4
4,product_length_cm,2,0,0,0
5,product_height_cm,2,0,0,0
6,product_width_cm,2,0,0,0


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,0,30,25,30
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,0,30,25,30
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,0,30,25,30
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,0,30,25,30


In [6]:
# 2.5. Thứ tự ngày giờ
# Mục đích: Chỉ so sánh khi có đủ hai mốc; chưa sửa dữ liệu gốc.

order_date_columns = ["order_purchase_timestamp", "order_approved_at",
                      "order_delivered_carrier_date", "order_delivered_customer_date",
                      "order_estimated_delivery_date"]
time_rules = {
    "flag_approval_before_purchase": ("order_approved_at", "order_purchase_timestamp"),
    "flag_carrier_before_purchase": ("order_delivered_carrier_date", "order_purchase_timestamp"),
    "flag_delivery_before_carrier": ("order_delivered_customer_date", "order_delivered_carrier_date"),
    "flag_delivery_before_purchase": ("order_delivered_customer_date", "order_purchase_timestamp"),
}
dates = data["orders"][order_date_columns].apply(
    pd.to_datetime, format="%Y-%m-%d %H:%M:%S", errors="coerce")
report = []
for flag, (later, earlier) in time_rules.items():
    comparable = dates[[later, earlier]].notna().all(axis=1)
    report.append({"check": flag, "comparable_rows": comparable.sum(),
                   "abnormal_rows": (comparable & dates[later].lt(dates[earlier])).sum()})
display(pd.DataFrame(report))


,check,comparable_rows,abnormal_rows
0,flag_approval_before_purchase,99281,0
1,flag_carrier_before_purchase,97658,166
2,flag_delivery_before_carrier,96475,23
3,flag_delivery_before_purchase,96476,0


## 3. Làm sạch và đánh dấu bất thường

`data` giữ dữ liệu gốc; `cleaned` là bản làm việc. Không điền ngày giao thiếu bằng ngày dự kiến.
Thông số số học thiếu vẫn để missing; khi làm ML, cách điền thiếu sẽ được học từ tập train.
Các cột review văn bản được giữ nguyên; cờ `has_comment_*` nhận biết nội dung rỗng.


In [7]:
# 3.1. Chuẩn hóa chuỗi
# Mục đích: Bỏ khoảng trắng thừa và chuyển chuỗi rỗng thành missing, trừ văn bản review.

cleaned = {name: df.copy(deep=True) for name, df in data.items()}
review_text = {"review_comment_title", "review_comment_message"}
report = []
for name, df in cleaned.items():
    trimmed = new_missing = 0
    for col in df.select_dtypes(include="string").columns:
        if name == "reviews" and col in review_text:
            continue
        source = df[col]
        normalized = source.str.strip().replace("", pd.NA)
        trimmed += source.ne(source.str.strip()).fillna(False).sum()
        new_missing += (source.notna() & normalized.isna()).sum()
        df[col] = normalized
    report.append({"table": name, "trimmed_cells": trimmed, "new_missing_cells": new_missing})
display(pd.DataFrame(report))


,table,trimmed_cells,new_missing_cells
0,customers,0,0
1,orders,0,0
2,items,0,0
3,payments,0,0
4,reviews,0,0
5,products,0,0
6,sellers,0,0
7,geolocation,1,0
8,translation,0,0


In [8]:
# 3.2. Tọa độ tra cứu
# Mục đích: Bỏ dòng trùng hoàn toàn rồi lấy tọa độ trung vị theo mã bưu chính.

geolocation_unique = cleaned["geolocation"].drop_duplicates().reset_index(drop=True).copy()
geo_work = geolocation_unique.copy()
for col in ["geolocation_lat", "geolocation_lng"]:
    geo_work[col] = pd.to_numeric(geo_work[col], errors="coerce")

valid_coordinates = (geo_work[zip_column].notna()
                     & geo_work["geolocation_lat"].between(-90, 90)
                     & geo_work["geolocation_lng"].between(-180, 180))
geolocation_lookup = geo_work.loc[valid_coordinates].groupby(zip_column, as_index=False).agg(
    latitude=("geolocation_lat", "median"), longitude=("geolocation_lng", "median"),
    source_row_count=("geolocation_lat", "size"))

print("Dòng trùng đã loại:", len(cleaned["geolocation"]) - len(geolocation_unique))
print("Dòng không dùng được do tọa độ/mã:", (~valid_coordinates).sum())
print("Số mã bưu chính:", len(geolocation_lookup))


Dòng trùng đã loại: 261831
Dòng không dùng được do tọa độ/mã: 0
Số mã bưu chính: 19015


In [9]:
# 3.3. Chuyển kiểu ngày giờ
# Mục đích: Chuyển ngày ở orders, items, reviews và báo cáo lỗi chuyển đổi.

date_fields = {
    "orders": order_date_columns,
    "items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}
date_report = []
for name, columns in date_fields.items():
    for col in columns:
        source = data[name][col].str.strip().replace("", pd.NA)
        converted = pd.to_datetime(source, format="%Y-%m-%d %H:%M:%S", errors="coerce")
        cleaned[name][col] = converted
        date_report.append({"table": name, "column": col,
                            "invalid_date_rows": (source.notna() & converted.isna()).sum(),
                            "missing_after": converted.isna().sum()})
display(pd.DataFrame(date_report))

orders = cleaned["orders"]
for flag, (later, earlier) in time_rules.items():
    comparable = orders[[later, earlier]].notna().all(axis=1)
    orders[flag] = orders[later].lt(orders[earlier]).astype("boolean").where(comparable)
orders["flag_delivered_missing_date"] = (
    orders["order_status"].eq("delivered") & orders["order_delivered_customer_date"].isna())
flags = list(time_rules) + ["flag_delivered_missing_date"]
display(orders[flags].sum().rename_axis("flag").to_frame("flagged_rows"))


,table,column,invalid_date_rows,missing_after
0,orders,order_purchase_timestamp,0,0
1,orders,order_approved_at,0,160
2,orders,order_delivered_carrier_date,0,1783
3,orders,order_delivered_customer_date,0,2965
4,orders,order_estimated_delivery_date,0,0
5,items,shipping_limit_date,0,0
6,reviews,review_creation_date,0,0
7,reviews,review_answer_timestamp,0,0


,flagged_rows
flag,
flag_approval_before_purchase,0
flag_carrier_before_purchase,166
flag_delivery_before_carrier,23
flag_delivery_before_purchase,0
flag_delivered_missing_date,8


In [10]:
# 3.4. Chuyển kiểu số
# Mục đích: Chuẩn hóa tiền, số kỳ, thông số sản phẩm và điểm review.

numeric_fields = {
    "items": ["price", "freight_value"],
    "payments": ["payment_value", "payment_installments"],
    "products": product_numeric,
    "reviews": ["review_score"],
}
numeric_report = []
for name, columns in numeric_fields.items():
    for col in columns:
        source = cleaned[name][col]
        converted = pd.to_numeric(source, errors="coerce")
        cleaned[name][col] = converted
        numeric_report.append({"table": name, "column": col,
                               "invalid_numeric_rows": (source.notna() & converted.isna()).sum(),
                               "missing_after": converted.isna().sum()})
display(pd.DataFrame(numeric_report))

payments, products, reviews = cleaned["payments"], cleaned["products"], cleaned["reviews"]
installments = payments["payment_installments"]
valid = installments.notna() & installments.ge(1) & installments.mod(1).eq(0)
payments["flag_invalid_installments"] = installments.notna() & ~valid
payments["payment_installments_valid"] = installments.where(valid).astype("Int64")

weight = products["product_weight_g"]
products["flag_invalid_weight"] = weight.notna() & weight.le(0)
products["product_weight_g_valid"] = weight.where(weight.gt(0))

score = reviews["review_score"]
valid = score.notna() & score.between(1, 5) & score.mod(1).eq(0)
reviews["review_score_valid"] = score.where(valid).astype("Int64")
reviews["flag_invalid_score"] = score.notna() & ~valid
for col, flag in [("review_comment_title", "has_comment_title"),
                  ("review_comment_message", "has_comment_message")]:
    reviews[flag] = reviews[col].fillna("").str.strip().ne("")

print("Số kỳ không hợp lệ:", payments["flag_invalid_installments"].sum())
print("Cân nặng không hợp lệ:", products["flag_invalid_weight"].sum())
print("Điểm review không hợp lệ:", reviews["flag_invalid_score"].sum())


Số kỳ không hợp lệ: 2
Cân nặng không hợp lệ: 4
Điểm review không hợp lệ: 0


,table,column,invalid_numeric_rows,missing_after
0,items,price,0,0
1,items,freight_value,0,0
2,payments,payment_value,0,0
3,payments,payment_installments,0,0
4,products,product_name_lenght,0,610
5,products,product_description_lenght,0,610
6,products,product_photos_qty,0,610
7,products,product_weight_g,0,2
8,products,product_length_cm,0,2
9,products,product_height_cm,0,2


In [11]:
# 3.5. Nhãn danh mục
# Mục đích: Ưu tiên bản dịch; nếu chưa có thì giữ tên gốc; thiếu danh mục dùng unknown.

# Bỏ cột tạo thêm trước khi ghép để cell này có thể chạy lại.
products = cleaned["products"].drop(columns=["product_category_name_english",
    "product_category_name_clean", "category_label_source"], errors="ignore")
products = products.merge(cleaned["translation"], on="product_category_name",
                          how="left", validate="many_to_one")
products["product_category_name_clean"] = (products["product_category_name_english"]
    .fillna(products["product_category_name"]).fillna("unknown"))
products["category_label_source"] = "missing_category"
products.loc[products["product_category_name"].notna(), "category_label_source"] = "original_name"
products.loc[products["product_category_name_english"].notna(),
             "category_label_source"] = "english_translation"
cleaned["products"] = products
display(products["category_label_source"].value_counts().to_frame("product_count"))


,product_count
category_label_source,
english_translation,32328
missing_category,610
original_name,13


In [12]:
# 3.6. Kiểm tra sau làm sạch
# Mục đích: Xác nhận không mất bản ghi, không thiếu/lặp khóa và tiền được bảo toàn.

report = []
for name, keys in key_columns.items():
    df = cleaned[name]
    report.append({"table": name, "rows_before": len(data[name]), "rows_after": len(df),
                   "missing_key_rows": df[keys].isna().any(axis=1).sum(),
                   "duplicate_key_rows": df.duplicated(keys, keep=False).sum()})
validation_report = pd.DataFrame(report)
display(validation_report)
assert validation_report["rows_before"].eq(validation_report["rows_after"]).all(), "Thay đổi số dòng!"
assert validation_report["missing_key_rows"].eq(0).all(), "Thiếu khóa!"
assert validation_report["duplicate_key_rows"].eq(0).all(), "Lặp khóa!"
assert geolocation_unique.duplicated().sum() == 0
assert geolocation_lookup[zip_column].notna().all() & geolocation_lookup[zip_column].is_unique
assert len(geolocation_lookup) == geolocation_unique[zip_column].dropna().nunique()

for name, col in [("items", "price"), ("items", "freight_value"), ("payments", "payment_value")]:
    before = pd.to_numeric(data[name][col], errors="coerce").sum()
    assert abs(before - cleaned[name][col].sum()) < 0.01, f"Tổng {name}.{col} thay đổi!"
print("Đạt kiểm tra số dòng, khóa, tọa độ tra cứu và tổng tiền.")


Đạt kiểm tra số dòng, khóa, tọa độ tra cứu và tổng tiền.


,table,rows_before,rows_after,missing_key_rows,duplicate_key_rows
0,customers,99441,99441,0,0
1,orders,99441,99441,0,0
2,items,112650,112650,0,0
3,payments,103886,103886,0,0
4,reviews,99224,99224,0,0
5,products,32951,32951,0,0
6,sellers,3095,3095,0,0
7,translation,71,71,0,0


In [13]:
# 4. Kiểm tra và xuất dữ liệu sạch
# Mục đích: Xuất 8 bảng sạch và 2 bảng tọa độ; không tạo bảng phân tích.

export_tables = {f"{name}_clean": df for name, df in cleaned.items() if name != "geolocation"}
export_tables["geolocation_unique"] = geolocation_unique
export_tables["geolocation_lookup"] = geolocation_lookup

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
export_report = []
for name, df in export_tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    path.write_text(df.to_csv(index=False), encoding="utf-8-sig")
    export_report.append({"file": path.name, "rows": len(df), "columns": len(df.columns)})
display(pd.DataFrame(export_report))
print("Đã xuất", len(export_tables), "bảng vào", OUTPUT_DIR.resolve())


Đã xuất 10 bảng vào /workspace/scratch/77261ae7e892/notebook_edit/validation_clean_only/data/processed


,file,rows,columns
0,customers_clean.csv,99441,5
1,orders_clean.csv,99441,13
2,items_clean.csv,112650,7
3,payments_clean.csv,103886,7
4,reviews_clean.csv,99224,11
5,products_clean.csv,32951,14
6,sellers_clean.csv,3095,4
7,translation_clean.csv,71,2
8,geolocation_unique.csv,738332,5
9,geolocation_lookup.csv,19015,4


## Cách sử dụng kết quả

- `customers_clean`, `orders_clean`, `items_clean`, `payments_clean`, `reviews_clean`, `products_clean`, `sellers_clean`, `translation_clean`: bảng sau chuẩn hóa.
- `geolocation_unique`: tọa độ sau bỏ dòng trùng hoàn toàn.
- `geolocation_lookup`: một mã bưu chính một dòng, dùng tọa độ trung vị làm vị trí đại diện.

Các cột bổ sung chỉ phục vụ chất lượng/chuẩn hóa: cờ ngày bất thường, số kỳ hợp lệ, cân nặng hợp lệ, điểm hợp lệ, cờ có văn bản và tên danh mục sạch. Missing số/ngày được giữ nếu chưa có căn cứ điền.

Dữ liệu thống kê mô tả nghiệp vụ, nhãn `is_late`, tổng hợp mặt hàng/thanh toán/review và bảng phân tích sẽ được tạo ở notebook khác.
Nếu từng chạy bản cũ, các file phân tích cũ trong `data/processed` vẫn còn; notebook này chỉ xuất 10 bảng kể trên, không tự xóa file khác.
Khi đọc lại CSV, khai báo kiểu chuỗi cho ID/mã bưu chính và chuyển lại các cột ngày/boolean khi cần.
